**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Graph Signal Processing & GNNs

A flagship IEEE-SPS research area with almost no accessible teaching material: signals that live on **networks** — sensor grids, social graphs, molecules, power grids. Four sessions: the graph Laplacian gives graphs a Fourier transform, filters, and sampling theory — and message-passing GNNs drop out as learned graph filters. The oracle throughout: on a ring graph, everything must reduce to classical DSP.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3 (eigendecomposition — this course is its victory lap).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (classical Fourier, for the reduction check).
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) for Session 4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def laplacian(A):
    return np.diag(A.sum(1)) - A

# our running graph: a random sensor network (geometric graph)
n_nodes = 80
pos = rng.random((n_nodes, 2))
D2 = ((pos[:, None] - pos[None]) ** 2).sum(-1)
A = ((D2 < 0.045) & (D2 > 0)).astype(float)
L = laplacian(A)
lam, U = np.linalg.eigh(L)                          # the graph's "frequencies" and "Fourier basis"

def draw(signal, title="", ax=None):
    if ax is None: fig, ax = plt.subplots(figsize=(3.6, 3.2))
    for i, j in zip(*np.nonzero(np.triu(A))):
        ax.plot(*zip(pos[i], pos[j]), "k-", linewidth=0.3, alpha=0.4)
    sc = ax.scatter(*pos.T, c=signal, s=45, cmap="coolwarm")
    ax.set_title(title, fontsize=9); ax.axis("off")
    return sc

---
### 🕐 Session 1 of 4 — *The Graph Laplacian & Graph Fourier Transform* (~40 min)
**Goal:** give any graph a frequency axis; verify it reduces to the DFT on a ring.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (filtering on graphs).

---

## 2. Frequency Without Time

💡 **Intuition.** What does 'frequency' mean with no time axis? **Smoothness with respect to the edges.** The Laplacian quadratic form $x^T L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges — so Laplacian eigenvectors, ordered by eigenvalue, are the graph's own harmonics: $\lambda \approx 0$ ⇒ smooth (neighbors agree), large $\lambda$ ⇒ oscillatory (neighbors alternate). The **graph Fourier transform** is just analysis in this eigenbasis: $\hat{x} = U^T x$ — [Hilbert-space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) change of basis, with the graph choosing the basis.

In [2]:
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, k in zip(axes, [0, 1, 4, 60]):
    draw(U[:, k], f"eigenvector {k}: λ={lam[k]:.2f}", ax)
plt.suptitle("the graph's own harmonics: smooth → oscillatory as λ grows", y=1.03)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2968206/3381734998.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [3]:
# ORACLE: on a RING graph, the Laplacian eigenvalues must be the classical DFT frequencies
N = 32
ring = np.zeros((N, N))
for i in range(N): ring[i, (i+1) % N] = ring[i, (i-1) % N] = 1
lam_ring = np.sort(np.linalg.eigvalsh(laplacian(ring)))
lam_theory = np.sort(2 - 2*np.cos(2*np.pi*np.arange(N)/N))   # from the DFT diagonalization
print("max |ring Laplacian eigs − 2−2cos(2πk/N)| =", np.abs(lam_ring - lam_theory).max())
assert np.abs(lam_ring - lam_theory).max() < 1e-9
print("→ classical DSP is the special case: the ring's Fourier basis diagonalizes its Laplacian")

max |ring Laplacian eigs − 2−2cos(2πk/N)| = 3.552713678800501e-15
→ classical DSP is the special case: the ring's Fourier basis diagonalizes its Laplacian


---
### 🕐 Session 2 of 4 — *Filtering on Graphs* (~40 min)
**Goal:** denoise a sensor field with a graph low-pass; make it local with polynomial filters.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sampling).

---

## 3. Graph Filters

💡 **Intuition.** A graph filter scales each harmonic: $y = U h(\Lambda) U^T x$ — design $h(\lambda)$ exactly like a [filter response](./Filter_Design.ipynb), with $\lambda$ replacing $\omega$. The practical twist: eigendecomposition is $O(n^3)$, but a **polynomial** filter $h(L) = \sum_k c_k L^k$ needs only matrix-vector products — and $L^k x$ touches only $k$-hop neighbors, so polynomial order = *filter locality*. That locality is the seed GNNs grow from.

In [4]:
# denoise a smooth temperature field over the sensor network
x_clean = U[:, :4] @ (rng.standard_normal(4) * [3, 2, 1.5, 1])   # smooth by construction
x_noisy = x_clean + 0.6 * rng.standard_normal(n_nodes)

h = 1.0 / (1.0 + 4.0 * lam)                                  # graph low-pass (Tikhonov)
x_filt = U @ (h * (U.T @ x_noisy))

# local polynomial approximation of the same filter (Chebyshev-lite: least-squares fit)
Vand = np.vander(lam, 6, increasing=True)
c, *_ = np.linalg.lstsq(Vand, h, rcond=None)
x_poly = np.zeros(n_nodes); Lk_x = x_noisy.copy()
for k in range(6):
    x_poly += c[k] * Lk_x
    Lk_x = L @ Lk_x

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (sig, t) in zip(axes, [(x_noisy, "noisy sensors"), (x_filt, "spectral low-pass"),
                                (x_poly, "5-hop polynomial (no eig needed)")]):
    draw(sig, t, ax)
plt.tight_layout(); plt.show()
for name, xh in [("spectral", x_filt), ("polynomial", x_poly)]:
    print(f"{name:10s} SNR gain: {10*np.log10(np.var(x_noisy-x_clean)/np.var(xh-x_clean)):.1f} dB")

spectral   SNR gain: 7.6 dB
polynomial SNR gain: 8.6 dB


/tmp/ipykernel_2968206/2748018384.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *Sampling on Graphs* (~35 min)
**Goal:** which sensors can you afford to lose? Bandlimited recovery from a subset of nodes.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (GNNs).

---

## 4. Nyquist for Networks

💡 **Intuition.** If a graph signal is **bandlimited** — lives in the span of the first $K$ harmonics — then $K$ well-chosen node readings determine *all* $n$: solve the little least-squares system in the known coefficients ([Compressed Sensing's](./Compressed_Sensing.ipynb) logic, subspace version). 'Well-chosen' matters exactly like array geometry: sample nodes that make the harmonics distinguishable, not clustered clones of each other.

In [5]:
K, m = 4, 12
x_band = x_clean                                             # bandlimited by construction (K=4)
sel = rng.choice(n_nodes, m, replace=False)                  # random sensor subset
coef, *_ = np.linalg.lstsq(U[sel, :K], x_band[sel], rcond=None)
x_rec = U[:, :K] @ coef
print(f"recover all {n_nodes} nodes from {m} sensors: max error {np.abs(x_rec - x_band).max():.2e}")
assert np.abs(x_rec - x_band).max() < 1e-8

# and the failure mode: measure fewer than K nodes → underdetermined
sel_bad = sel[:3]
coef_bad, *_ = np.linalg.lstsq(U[sel_bad, :K], x_band[sel_bad], rcond=None)
print(f"with only 3 < K sensors: max error {np.abs(U[:, :K] @ coef_bad - x_band).max():.2f}  (aliasing, graph edition)")

recover all 80 nodes from 12 sensors: max error 2.22e-15
with only 3 < K sensors: max error 0.13  (aliasing, graph edition)


---
### 🕐 Session 4 of 4 — *Message Passing = Learned Graph Filters* (~40 min)
**Goal:** build a GCN from scratch; classify nodes; see it as Session 2 with trained coefficients.
**Builds on:** Session 3; [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

---

## 5. GNNs, Demystified

💡 **Intuition.** A graph-convolution layer is: *average your neighbors (a fixed 1-hop low-pass $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$), then apply a learned linear map and a nonlinearity*. Stack $k$ layers ⇒ $k$-hop receptive field — precisely Session 2's polynomial filters with coefficients chosen by gradient descent. It's the [CNN story](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) (weight sharing + locality) generalized to irregular neighborhoods.

In [6]:
import torch
import torch.nn as nn
torch.manual_seed(0)

# two-community node classification (a planted partition on top of geometry)
comm = (pos[:, 0] + 0.25*rng.standard_normal(n_nodes) > 0.5).astype(int)
A2 = A.copy()
for i in range(n_nodes):                                   # densify within communities
    same = np.where((comm == comm[i]) & (np.arange(n_nodes) != i))[0]
    for j in rng.choice(same, 2): A2[i, j] = A2[j, i] = 1
At = A2 + np.eye(n_nodes)
Dh = np.diag(1/np.sqrt(At.sum(1)))
Ahat = torch.tensor(Dh @ At @ Dh, dtype=torch.float32)

feats = torch.tensor(np.stack([rng.standard_normal(n_nodes),  # useless feature
                               comm + 1.2*rng.standard_normal(n_nodes)], 1),  # noisy hint
                     dtype=torch.float32)
labels = torch.tensor(comm)
train_mask = torch.zeros(n_nodes, dtype=bool); train_mask[rng.choice(n_nodes, 10, replace=False)] = True

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1, self.W2 = nn.Linear(2, 16), nn.Linear(16, 2)
    def forward(self, X):
        H = torch.relu(self.W1(Ahat @ X))       # aggregate neighbors → transform → nonlinearity
        return self.W2(Ahat @ H)

class MLP(nn.Module):                            # ablation: same net, NO graph
    def __init__(self):
        super().__init__()
        self.W1, self.W2 = nn.Linear(2, 16), nn.Linear(16, 2)
    def forward(self, X): return self.W2(torch.relu(self.W1(X)))

for name, model in [("GCN (uses edges)", GCN()), ("MLP (ignores edges)", MLP())]:
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    for step in range(300):
        loss = nn.functional.cross_entropy(model(feats)[train_mask], labels[train_mask])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        acc = (model(feats).argmax(1) == labels)[~train_mask].float().mean()
    print(f"{name:20s}: {acc:.1%} accuracy on unseen nodes (10 labeled examples!)")

GCN (uses edges)    : 88.6% accuracy on unseen nodes (10 labeled examples!)
MLP (ignores edges) : 41.4% accuracy on unseen nodes (10 labeled examples!)


Ten labels classify eighty nodes because the graph *propagates* them — message passing is label smoothing through a learned low-pass. (Also visible here: stack too many layers and everything averages toward mush — *oversmoothing*, the graph version of over-aggressive low-pass filtering.)

## 6. Conclusion

The Laplacian gives every network a Fourier basis (reducing to the DFT on a ring — verified); filters are functions of $L$, made local by polynomials; bandlimited signals need only $K$ good sensors; and GNNs are those polynomial filters with learned coefficients. Classical DSP was the special case all along.

---
## Where next

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the eigen-machinery, if it felt fast.
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — the regular-grid special case.
- [Statistical SP](./Statistical_Signal_Processing.ipynb) — stochastic graph signals are an open research door.